# Ejemplos de Uso de la API REST

**Objetivo:** Demostrar el uso de todos los endpoints de la API REST del sistema
**Duración estimada:** 25 minutos

---

## Contenido

1. [Setup](#setup)
2. [Configuración del Cliente HTTP](#configuracion)
3. [Health Check](#health-check)
4. [Endpoint de Análisis Completo](#analisis-completo)
5. [Endpoint de Detección de Patrones](#deteccion-patrones)
6. [Endpoint de Detección de Estructuras](#deteccion-estructuras)
7. [Endpoint de Visualización](#visualizacion)
8. [Endpoint de Exportación](#exportacion)
9. [Gestión de Algoritmos (CRUD)](#crud-algoritmos)
10. [Manejo de Errores](#manejo-errores)

---

## 1. Setup

In [ ]:
import sys
import json
import time

# httpx es el cliente HTTP recomendado para FastAPI
try:
    import httpx
    HTTPX_DISPONIBLE = True
except ImportError:
    HTTPX_DISPONIBLE = False
    print("httpx no disponible. Instalar con: pip install httpx")

# requests como alternativa
try:
    import requests
    REQUESTS_DISPONIBLE = True
except ImportError:
    REQUESTS_DISPONIBLE = False

# Verificar si el servidor está disponible antes de ejecutar
BASE_URL = "http://localhost:8000"
API_V1 = f"{BASE_URL}/api/v1"

def verificar_servidor(url):
    """Verifica si el servidor FastAPI está corriendo."""
    try:
        if HTTPX_DISPONIBLE:
            resp = httpx.get(f"{url}/api/v1/health", timeout=3.0)
            return resp.status_code == 200
        elif REQUESTS_DISPONIBLE:
            resp = requests.get(f"{url}/api/v1/health", timeout=3.0)
            return resp.status_code == 200
    except Exception:
        return False
    return False

SERVIDOR_DISPONIBLE = verificar_servidor(BASE_URL)
if SERVIDOR_DISPONIBLE:
    print(f"Servidor disponible en: {BASE_URL}")
else:
    print(f"ADVERTENCIA: Servidor no disponible en {BASE_URL}")
    print("Para iniciar el servidor: python scripts/run_dev.py")
    print("Los ejemplos se mostrarán con respuestas simuladas donde aplique.")

---

## 2. Configuración del Cliente HTTP

In [ ]:
def hacer_peticion(metodo, endpoint, datos=None, params=None):
    """
    Función helper para hacer peticiones a la API.
    
    Maneja tanto httpx como requests como fallback.
    """
    url = f"{API_V1}{endpoint}"
    
    if not SERVIDOR_DISPONIBLE:
        print(f"[SIMULADO] {metodo.upper()} {url}")
        return None
    
    try:
        if HTTPX_DISPONIBLE:
            with httpx.Client(timeout=30.0) as client:
                if metodo.lower() == "get":
                    resp = client.get(url, params=params)
                elif metodo.lower() == "post":
                    resp = client.post(url, json=datos, params=params)
                elif metodo.lower() == "delete":
                    resp = client.delete(url)
                else:
                    raise ValueError(f"Método no soportado: {metodo}")
        elif REQUESTS_DISPONIBLE:
            if metodo.lower() == "get":
                resp = requests.get(url, params=params, timeout=30)
            elif metodo.lower() == "post":
                resp = requests.post(url, json=datos, timeout=30)
            elif metodo.lower() == "delete":
                resp = requests.delete(url, timeout=30)
        else:
            print("ERROR: No hay cliente HTTP disponible")
            return None
        
        return resp
    except Exception as e:
        print(f"Error en petición {metodo.upper()} {url}: {e}")
        return None

def mostrar_respuesta(resp, titulo="RESPUESTA"):
    """Muestra una respuesta HTTP de forma legible."""
    if resp is None:
        return
    print(f"\n{titulo}:")
    print(f"  Status: {resp.status_code}")
    try:
        datos = resp.json()
        print(f"  Body: {json.dumps(datos, indent=4, ensure_ascii=False)[:800]}")
    except Exception:
        print(f"  Body: {resp.text[:200]}")

print("Cliente HTTP configurado")

---

## 3. Health Check

In [ ]:
print("ENDPOINT: GET /health")
print("Propósito: Verificar que el sistema está operativo")

resp_health = hacer_peticion("GET", "/health")
mostrar_respuesta(resp_health, "Health Check")

# Respuesta esperada (referencia)
RESPUESTA_HEALTH_ESPERADA = {
    "status": "healthy",
    "version": "1.0.0",
    "database": "connected",
    "cache": "connected"
}
print("\nEstructura de respuesta esperada:")
print(json.dumps(RESPUESTA_HEALTH_ESPERADA, indent=2))

---

## 4. Endpoint de Análisis Completo

In [ ]:
print("ENDPOINT: POST /analysis/analyze")
print("Propósito: Análisis completo de complejidad de un algoritmo")

# Ejemplo 1: Análisis básico
REQUEST_ANALISIS_BASICO = {
    "algorithm_code": """
algorithm bubbleSort(A[], n)
begin
    for i <- 1 to n - 1 do
        for j <- 1 to n - i do
            if (A[j] > A[j + 1]) then
                temp <- A[j]
                A[j] <- A[j + 1]
                A[j + 1] <- temp
            end
        end
    end
end
""",
    "include_patterns": True,
    "include_structures": True,
    "include_line_by_line": False
}

print("Request Body:")
print(json.dumps({**REQUEST_ANALISIS_BASICO, "algorithm_code": "..."}, indent=2))

resp_analisis = hacer_peticion("POST", "/analysis/analyze", REQUEST_ANALISIS_BASICO)
mostrar_respuesta(resp_analisis, "Análisis Completo")

# Estructura de respuesta de referencia
RESPUESTA_ANALISIS_REFERENCIA = {
    "success": True,
    "data": {
        "algorithm_name": "bubbleSort",
        "big_o": "O(n^2)",
        "omega": "O(n)",
        "theta": "O(n^2)",
        "space_complexity": "O(1)",
        "patterns": {
            "primary_pattern": "SORTING",
            "primary_confidence": 0.85
        },
        "structures": ["ARRAY"],
        "execution_time_ms": 12.5
    }
}
print("\nEstructura de respuesta esperada:")
print(json.dumps(RESPUESTA_ANALISIS_REFERENCIA, indent=2))

### Análisis con Opciones Avanzadas

In [ ]:
# Ejemplo 2: Análisis con ecuación de recurrencia
REQUEST_ANALISIS_AVANZADO = {
    "algorithm_code": """
algorithm mergeSort(A[], p, r)
begin
    if (p < r) then
        q <- floor((p + r) / 2)
        call mergeSort(A, p, q)
        call mergeSort(A, q + 1, r)
    end
end
""",
    "include_patterns": True,
    "include_structures": True,
    "include_recurrence": True,
    "include_line_by_line": True
}

print("\nRequest con opciones avanzadas (merge sort):")
resp_avanzado = hacer_peticion("POST", "/analysis/analyze", REQUEST_ANALISIS_AVANZADO)
mostrar_respuesta(resp_avanzado, "Análisis Avanzado")

---

## 5. Endpoint de Detección de Patrones

In [ ]:
print("ENDPOINT: POST /patterns/detect")
print("Propósito: Detectar patrones algorítmicos en el código")

REQUEST_PATRONES = {
    "algorithm_code": """
algorithm knapsack(weights[], values[], n, capacity)
begin
    for i <- 1 to n do
        for w <- 0 to capacity do
            if (weights[i] <= w) then
                dp[i][w] <- dp[i-1][w-weights[i]] + values[i]
            else
                dp[i][w] <- dp[i-1][w]
            end
        end
    end
    return dp[n][capacity]
end
""",
    "min_confidence": 0.3,
    "return_all_patterns": True
}

resp_patrones = hacer_peticion("POST", "/patterns/detect", REQUEST_PATRONES)
mostrar_respuesta(resp_patrones, "Detección de Patrones")

RESPUESTA_PATRONES_REFERENCIA = {
    "success": True,
    "data": {
        "primary_pattern": "DYNAMIC_PROGRAMMING",
        "primary_confidence": 0.78,
        "all_patterns": [
            {"pattern": "DYNAMIC_PROGRAMMING", "confidence": 0.78},
            {"pattern": "BRUTE_FORCE", "confidence": 0.35}
        ],
        "summary": "El algoritmo usa programación dinámica con tabla dp de tamaño n*capacity"
    }
}
print("\nEstructura de respuesta esperada:")
print(json.dumps(RESPUESTA_PATRONES_REFERENCIA, indent=2))

---

## 6. Endpoint de Detección de Estructuras

In [ ]:
print("ENDPOINT: POST /structures/identify")
print("Propósito: Identificar estructuras de datos usadas en el algoritmo")

REQUEST_ESTRUCTURAS = {
    "algorithm_code": """
algorithm bfs(grafo[][], inicio, n)
begin
    for i <- 1 to n do
        visitado[i] <- false
    end
    cola[1] <- inicio
    frente <- 1
    fin_cola <- 1
    visitado[inicio] <- true
    while (frente <= fin_cola) do
        nodo <- cola[frente]
        frente <- frente + 1
        for vecino <- 1 to n do
            if (grafo[nodo][vecino] = 1 and not visitado[vecino]) then
                visitado[vecino] <- true
                fin_cola <- fin_cola + 1
                cola[fin_cola] <- vecino
            end
        end
    end
end
"""
}

resp_estructuras = hacer_peticion("POST", "/structures/identify", REQUEST_ESTRUCTURAS)
mostrar_respuesta(resp_estructuras, "Detección de Estructuras")

---

## 7. Endpoint de Visualización

In [ ]:
print("ENDPOINT: POST /visualization/generate")
print("Propósito: Generar visualizaciones gráficas del algoritmo")

REQUEST_VISUALIZACION = {
    "algorithm_code": """
algorithm fibonacci(n)
begin
    if (n <= 1) then
        return n
    end
    return fibonacci(n - 1) + fibonacci(n - 2)
end
""",
    "visualization_type": "recursion_tree",
    "max_depth": 4,
    "output_format": "mermaid"
}

resp_viz = hacer_peticion("POST", "/visualization/generate", REQUEST_VISUALIZACION)
mostrar_respuesta(resp_viz, "Visualización Generada")

print("\nTipos de visualización disponibles:")
tipos_viz = {
    "recursion_tree": "Árbol de recursión para algoritmos recursivos",
    "execution_flow": "Diagrama de flujo de ejecución",
    "graph": "Grafo de estructura del algoritmo",
}
for tipo, descripcion in tipos_viz.items():
    print(f"  - {tipo}: {descripcion}")

print("\nFormatos de salida disponibles: svg, png, dot, mermaid")

---

## 8. Endpoint de Exportación

In [ ]:
print("ENDPOINT: POST /export/generate")
print("Propósito: Exportar resultados de análisis a diferentes formatos")

REQUEST_EXPORTACION = {
    "algorithm_id": "ejemplo-merge-sort",
    "format": "markdown",
    "sections": ["complexity", "patterns", "structures"],
    "include_code": True
}

resp_export = hacer_peticion("POST", "/export/generate", REQUEST_EXPORTACION)
mostrar_respuesta(resp_export, "Exportación")

print("\nFormatos de exportación disponibles:")
formatos = {
    "json": "Datos estructurados en JSON",
    "markdown": "Reporte en Markdown (.md)",
    "pdf": "Reporte en PDF (requiere reportlab)",
    "html": "Reporte en HTML",
    "excel": "Datos en hoja de cálculo (requiere openpyxl)",
    "csv": "Datos tabulares en CSV",
    "dot": "Grafo en formato DOT (Graphviz)",
    "mermaid": "Diagrama en formato Mermaid",
    "svg": "Gráfico vectorial SVG",
}
for fmt, desc in formatos.items():
    print(f"  - {fmt}: {desc}")

---

## 9. Gestión de Algoritmos (CRUD)

In [ ]:
print("ENDPOINTS: /algorithms - Gestión de Algoritmos")

# Crear un nuevo algoritmo
REQUEST_CREAR = {
    "name": "quicksort_demo",
    "code": """
algorithm quickSort(A[], low, high)
begin
    if (low < high) then
        pivot <- A[high]
        i <- low - 1
        for j <- low to high - 1 do
            if (A[j] <= pivot) then
                i <- i + 1
                temp <- A[i]
                A[i] <- A[j]
                A[j] <- temp
            end
        end
        temp <- A[i + 1]
        A[i + 1] <- A[high]
        A[high] <- temp
        call quickSort(A, low, i)
        call quickSort(A, i + 2, high)
    end
end
""",
    "category": "sorting",
    "tags": ["sorting", "divide_and_conquer", "in_place"]
}

print("POST /algorithms - Crear algoritmo:")
resp_crear = hacer_peticion("POST", "/algorithms", REQUEST_CREAR)
mostrar_respuesta(resp_crear, "Crear Algoritmo")

# Obtener algoritmo por ID
algoritmo_id = None
if resp_crear and resp_crear.status_code == 201:
    try:
        datos = resp_crear.json()
        algoritmo_id = datos.get("data", {}).get("id")
        print(f"\nAlgoritmo creado con ID: {algoritmo_id}")
    except Exception:
        pass

if algoritmo_id:
    print(f"\nGET /algorithms/{algoritmo_id} - Obtener algoritmo:")
    resp_get = hacer_peticion("GET", f"/algorithms/{algoritmo_id}")
    mostrar_respuesta(resp_get, "Obtener Algoritmo")

# Listar algoritmos
print("\nGET /algorithms - Listar algoritmos:")
resp_list = hacer_peticion("GET", "/algorithms", params={"page": 1, "limit": 5})
mostrar_respuesta(resp_list, "Lista de Algoritmos")

---

## 10. Manejo de Errores

In [ ]:
print("MANEJO DE ERRORES")
print("Casos de error comunes y sus respuestas")

CASOS_ERROR = [
    ("Pseudocódigo inválido", {
        "algorithm_code": "esto no es pseudocodigo valido !!!"
    }),
    ("Algoritmo vacío", {
        "algorithm_code": ""
    }),
    ("Formato de request inválido", {
        "codigo": "campo incorrecto"
    }),
]

for nombre_caso, request_data in CASOS_ERROR:
    print(f"\nCaso de error: '{nombre_caso}'")
    resp_error = hacer_peticion("POST", "/analysis/analyze", request_data)
    if resp_error:
        print(f"  Status HTTP: {resp_error.status_code}")
        try:
            datos = resp_error.json()
            print(f"  success: {datos.get('success', '?')}")
            if "error" in datos:
                print(f"  error.type: {datos['error'].get('type', '?')}")
                print(f"  error.message: {datos['error'].get('message', '?')[:60]}")
        except Exception:
            pass

# Estructura estándar de errores
print("\nEstructura estándar de respuesta de error:")
RESPUESTA_ERROR_REFERENCIA = {
    "success": False,
    "error": {
        "type": "ParserError",
        "message": "Error de sintaxis en el pseudocódigo",
        "details": {
            "line": 3,
            "column": 5,
            "expected": ["END", "FOR", "WHILE", "IF"]
        }
    }
}
print(json.dumps(RESPUESTA_ERROR_REFERENCIA, indent=2))

print("Para documentación completa de la API: GET /docs")
print("Para documentación ReDoc: GET /redoc")

---

## Proximos Pasos

- **complete_analysis_demo.ipynb**: Demostración completa del sistema sin API
- Documentación completa disponible en `docs/API_REFERENCE.md`
- Swagger UI disponible en `http://localhost:8000/docs` cuando el servidor está activo